# CSE476 CA1 — Personal Budget Assistant Agent

This notebook demonstrates a real agentic AI system for tracking a student's monthly budget.

The demonstration shows:

1. Tool calling
2. Multi-step plan-act execution
3. Memory across conversation turns
4. Decision making based on tool results

## Core tools
- `add_expense(item, amount, category)`
- `get_summary(category)`

The project follows the CSE476 CA1 requirement for an agent that calls tools, takes more than one step, and remembers earlier turns.

## 1. Install dependencies

Run this cell once if the required packages are not already installed.

In [1]:
!pip install groq python-dotenv


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Import the agent

`agent.py` contains the tools, session memory, Groq client, and the plan-act orchestration loop.

In [2]:
import json
from agent import run_agent

## 3. Initialize conversation memory

The conversation history is shared between all demonstrations. This allows the agent to remember information from earlier turns.

In [3]:
conversation_history = [
    {
        "role": "system",
        "content": """
        You are a Personal Budget Assistant agent.

        Use tools to interact with financial memory.

        When the user reports an expense, use add_expense.

        When the user asks about their balance or affordability,
        use get_summary before answering.

        Never invent financial data.

        For affordability questions:
        1. Retrieve the current balance.
        2. Compare the proposed expense with the remaining balance.
        3. Decide whether it is affordable.
        4. Explain the decision.

        Remember information from earlier turns.
        """
    }
]

# Demo 1 — Add multiple expenses

The user provides two expenses in one request. The agent should decide to call `add_expense` for each expense and then provide a final response.

In [4]:
run_agent(
    "I bought textbooks for 1200 under study and spent 300 on groceries.",
    conversation_history
)


USER PROMPT
I bought textbooks for 1200 under study and spent 300 on groceries.

------------------------------------------------------------
TRACE - TOOL CALL
------------------------------------------------------------
add_expense({'amount': 1200, 'category': 'study', 'item': 'textbooks'})

TRACE - TOOL RESULT
------------------------------------------------------------
Expense successfully added: textbooks = $1200.00, category = study.

------------------------------------------------------------
TRACE - TOOL CALL
------------------------------------------------------------
add_expense({'amount': 300, 'category': 'groceries', 'item': 'groceries'})

TRACE - TOOL RESULT
------------------------------------------------------------
Expense successfully added: groceries = $300.00, category = groceries.

AGENT RESPONSE
Got it! Your textbook purchase ($1,200) and groceries ($300) have been logged. If you’d like to check your remaining budget or see a summary of your spending, just let me 

'Got it! Your textbook purchase ($1,200) and groceries ($300) have been logged. If you’d like to check your remaining budget or see a summary of your spending, just let me know!'

### What this demonstrates

Expected agentic flow:

```text
User request
    ↓
Agent decides to use a tool
    ↓
add_expense(textbooks, 1200, study)
    ↓
Tool result returned to agent
    ↓
add_expense(groceries, 300, food)
    ↓
Tool result returned to agent
    ↓
Final response
```

The terminal output from `run_agent()` shows the actual tool calls and tool results.

# Demo 2 — Affordability decision using previous memory

The agent is now asked whether a $2,000 weekend trip is affordable. It must retrieve the current financial state rather than guessing.

In [6]:
run_agent(
    "Can I afford a 2000 weekend trip right now?",
    conversation_history
)


USER PROMPT
Can I afford a 2000 weekend trip right now?

------------------------------------------------------------
TRACE - TOOL CALL
------------------------------------------------------------
get_summary({})

TRACE - TOOL RESULT
------------------------------------------------------------
Monthly budget: $5000.00
Total spent: $1500.00
Remaining balance: $3500.00
Expenses: [{'item': 'textbooks', 'amount': 1200.0, 'category': 'study'}, {'item': 'groceries', 'amount': 300.0, 'category': 'groceries'}]

AGENT RESPONSE
Your current remaining balance is **$3,500.00**. A $2,000 weekend trip would leave you with **$1,500.00** after the trip.

**Affordability:** Yes, you can afford it. The trip costs less than the remaining balance, so you still have plenty left over for other expenses or savings. Let me know if you'd like to add the trip to your expense record or check anything else!


"Your current remaining balance is **$3,500.00**. A $2,000 weekend trip would leave you with **$1,500.00** after the trip.\n\n**Affordability:** Yes, you can afford it. The trip costs less than the remaining balance, so you still have plenty left over for other expenses or savings. Let me know if you'd like to add the trip to your expense record or check anything else!"

### Expected reasoning flow

The previous turn recorded:

- Textbooks: $1,200
- Groceries: $300
- Total spent: $1,500
- Monthly budget: $5,000
- Remaining balance: $3,500

The agent should call `get_summary()`, receive the current balance, compare the proposed $2,000 trip against the $3,500 remaining balance, and conclude that the trip fits within the monthly budget.

The important point is that the $1,200 and $300 expenses came from the earlier turn and were retained in session memory.

# Demo 3 — Category spending query

This test asks the agent to retrieve spending for the `study` category from the expenses already stored in memory.

In [7]:
run_agent(
    "How much have I spent on study?",
    conversation_history
)


USER PROMPT
How much have I spent on study?

------------------------------------------------------------
TRACE - TOOL CALL
------------------------------------------------------------
get_summary({'category': 'study'})

TRACE - TOOL RESULT
------------------------------------------------------------
Category: study
Total spent: $1200.00
Transactions: [{'item': 'textbooks', 'amount': 1200.0, 'category': 'study'}]

AGENT RESPONSE
You've spent **$1,200.00** on study‑related expenses.


"You've spent **$1,200.00** on study‑related expenses."

### What this demonstrates

The agent should call:

```text
get_summary(category="study")
```

The stored textbook expense should be included in the category total.

# Final — Inspect session memory

This cell makes the agent's stored session state visible for demonstration and viva purposes.

In [8]:
from agent import memory

print("Current Agent Memory:")
print(json.dumps(memory, indent=4))

Current Agent Memory:
{
    "expenses": [
        {
            "item": "textbooks",
            "amount": 1200.0,
            "category": "study"
        },
        {
            "item": "groceries",
            "amount": 300.0,
            "category": "groceries"
        }
    ],
    "monthly_budget": 5000.0
}


# Conclusion

The demonstration shows the required agentic behavior:

- The agent calls Python tools instead of only generating text.
- The orchestration loop allows multiple tool-call steps.
- Tool results are returned to the model before it produces its next response.
- Expense information is retained in session memory and used in later turns.
- The affordability question is answered using the retrieved financial state.